<a href="https://colab.research.google.com/github/JoelForson/ECON5200-Applied-Data-Analytics-in-Economics/blob/main/Lab%209/Lab_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Environment and Data Strategy
We will utilize statsmodels, sklearn, causalinference, and seaborn to analyze the lalonde_obs.csv dataset.

In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

# Load your dataset here (ensure lalonde.csv is uploaded to Colab or linked)
df = pd.read_csv('lalonde.csv')

# Step 2: The Observational Failure (Naive Comparison)
The first task is to reproduce the "Crisis of Observation" by calculating the raw difference in means.

In [4]:
# Naive Comparison
naive_diff = df[df.treat==1]['re78'].mean() - df[df.treat==0]['re78'].mean()
print(f"Naive Difference in Means: ${naive_diff:,.2f}")
# Expected Result: -$635.03

Naive Difference in Means: $-635.03


# Step 3: Propensity Score Estimation
Build a model to predict who chooses training by selecting confounders that influence both participation and earnings.

In [6]:
print("Available columns:")
print(df.columns.tolist())

Available columns:
['Unnamed: 0', 'treat', 'age', 'educ', 'black', 'hispan', 'married', 'nodegree', 're74', 're75', 're78']


In [7]:
# Define covariates
X = df[['age', 'educ', 'black', 'hispan', 'married', 'nodegree', 're74', 're75']]
y = df['treat']

# Fit Propensity Model
logit = LogisticRegression(solver='liblinear')
logit.fit(X, y)

# Generate Scores
df['pscore'] = logit.predict_proba(X)[:, 1]

# Step 4: The Matching Algorithm (Nearest Neighbor)
Implement 1:1 Nearest Neighbor matching with replacement. Because the pool of "good" controls is small, reusing matches minimizes bias

In [8]:
from sklearn.neighbors import NearestNeighbors

# Separate groups
treated = df[df.treat==1]
control = df[df.treat==0]

# Fit NN on Control scores
nbrs = NearestNeighbors(n_neighbors=1, algorithm='ball_tree').fit(control[['pscore']])

# Find matches for Treated scores
distances, indices = nbrs.kneighbors(treated[['pscore']])
matched_control = control.iloc[indices.flatten()]

# Construct Matched DataFrame
matched_df = pd.concat([treated, matched_control])

# Step 5: Assessing Balance and Estimating the Effect
Verify that the Standardized Mean Difference (SMD) for all covariates is less than 0.1. Finally, run the T-test on the matched sample to find the difference.

In [9]:
from scipy import stats

# T-test on raw data
diff = treated['re78'].mean() - control['re78'].mean()
t_stat, p_val = stats.ttest_ind(treated['re78'], control['re78'])

print(f"Raw Effect (Difference): ${diff:,.2f}")
print(f"P-value: {p_val:.4f}")


# Isolate the matched outcomes
matched_treated = matched_df[matched_df.treat==1]['re78']
matched_control = matched_df[matched_df.treat==0]['re78']

# Estimate the causal effect (T-test on matched data)
matched_diff = matched_treated.mean() - matched_control.mean()
t_stat, p_val = stats.ttest_ind(matched_treated, matched_control)


print(f"Recovered Effect (Matched Difference): ${matched_diff:,.2f}")
print(f"P-value: {p_val:.4f}")

Raw Effect (Difference): $-635.03
P-value: 0.3342
Recovered Effect (Matched Difference): $583.04
P-value: 0.4438
